## 0. 加载 Qwen 模型

本 Notebook 使用 **Qwen2.5-7B-Instruct**（通过 ModelScope 加载）作为真实 LLM 后端，替代原有的模拟 LLM。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

### 安装依赖（首次运行需要）

```bash
pip install modelscope torch transformers
```


In [ ]:
# ============================================================
# 加载 Qwen 模型（通过 ModelScope）
# ============================================================
# 如果没有 GPU 或显存不足，可将模型 ID 改为 Qwen/Qwen2.5-3B-Instruct
# ============================================================

import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer


class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装类"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测 GPU / CPU
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("[QwenLLM] 模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """调用模型进行对话"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """重置对话历史"""
        self.messages = []


# 初始化 QwenLLM 实例
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
print("\n模型就绪，可以开始使用了。")

# 项目二：智能研究助手

## 项目目标

构建一个能够协助研究的智能 Agent，能够：
- 分解研究任务
- 搜索和收集信息
- 分析和总结内容
- 生成研究报告

## 技术栈

- 任务规划：任务分解和分配
- 信息检索：模拟搜索
- 内容分析：文本处理
- 报告生成：结构化输出

---

## 1. 系统架构

```
┌─────────────────────────────────────────────────────────────┐
│                  智能研究助手                                │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐     │
│  │  任务分解    │───►│  信息收集    │───►│  内容分析    │     │
│  │  (Planner)  │    │  (Researcher)│    │  (Analyzer) │     │
│  └─────────────┘    └─────────────┘    └──────┬──────┘     │
│                                                │            │
│  ┌─────────────┐    ┌─────────────┐    ┌──────┴──────┐     │
│  │  报告输出    │◄───│  内容整合    │◄───│  结果汇总    │     │
│  │  (Report)   │    │  (Synthesizer)│   │  (Summary)  │     │
│  └─────────────┘    └─────────────┘    └─────────────┘     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 2. 实现代码

In [ ]:
from typing import List, Dict
import random

class ResearchTask:
    """研究任务"""
    
    def __init__(self, topic: str, subtasks: List[str] = None):
        self.topic = topic
        self.subtasks = subtasks or []
        self.results = {}
        self.status = "pending"
    
    def add_result(self, subtask: str, result: str):
        self.results[subtask] = result
    
    def is_complete(self) -> bool:
        return len(self.results) == len(self.subtasks)

class ResearchAssistant:
    """智能研究助手（使用 QwenLLM 进行分析和报告生成）"""
    
    def __init__(self, llm_model=None):
        self.tasks = []
        self.knowledge_base = self._init_knowledge_base()
        self.llm = llm_model  # QwenLLM 实例
    
    def _init_knowledge_base(self) -> Dict:
        """初始化知识库"""
        return {
            "AI Agent": [
                "AI Agent 是能够自主决策和执行任务的智能系统",
                "核心组件包括感知、推理、行动和记忆",
                "ReAct 架构结合了推理和行动",
                "多 Agent 系统可以协作完成复杂任务"
            ],
            "RAG": [
                "RAG 结合了检索和生成技术",
                "核心流程包括文档加载、分块、嵌入和检索",
                "向量数据库用于存储文档向量",
                "RAG 可以减少 LLM 的幻觉问题"
            ],
            "LangChain": [
                "LangChain 是 LLM 应用开发框架",
                "提供模型接口、提示词管理、链式调用",
                "支持记忆系统和 Agent 功能",
                "可以集成多种工具和数据库"
            ]
        }
    
    def plan_research(self, topic: str) -> ResearchTask:
        """规划研究任务（使用 QwenLLM 进行智能分解）"""
        print(f"\n规划研究: {topic}")
        
        if self.llm is not None:
            # 使用 QwenLLM 智能分解子任务
            system_prompt = (
                "你是一个研究规划专家。请将给定研究主题分解为 4 个具体的子任务，"
                "每个子任务用一行描述，不要加编号或其他格式。"
            )
            response = self.llm.chat(
                f"请为以下主题分解研究子任务：{topic}",
                system_prompt=system_prompt, max_new_tokens=300, temperature=0.7
            )
            subtasks = [line.strip() for line in response.strip().split("\n") if line.strip()][:4]
        else:
            # ---- 无模型时的备选方案 ----
            subtasks = [
                f"{topic} 的基本概念和定义",
                f"{topic} 的核心技术和方法",
                f"{topic} 的应用场景和案例",
                f"{topic} 的发展趋势和挑战"
            ]
        
        task = ResearchTask(topic, subtasks)
        self.tasks.append(task)
        
        print(f"分解为 {len(subtasks)} 个子任务:")
        for i, st in enumerate(subtasks, 1):
            print(f"  {i}. {st}")
        
        return task
    
    def research(self, task: ResearchTask) -> Dict:
        """执行研究（使用 QwenLLM 进行信息分析）"""
        print(f"\n开始研究: {task.topic}")
        print("="*60)
        
        for subtask in task.subtasks:
            print(f"\n研究子任务: {subtask}")
            
            # 收集信息
            info = self._collect_information(subtask)
            
            # 使用 QwenLLM 分析信息
            analysis = self._analyze_information(subtask, info)
            
            task.add_result(subtask, analysis)
            print(f"完成: {analysis[:80]}...")
        
        task.status = "completed"
        return task.results
    
    def _collect_information(self, query: str) -> List[str]:
        """收集信息"""
        results = []
        for topic, infos in self.knowledge_base.items():
            if any(keyword in query for keyword in topic.split()):
                results.extend(infos)
        
        if not results:
            results = [
                f"关于 '{query}' 的信息点 1",
                f"关于 '{query}' 的信息点 2",
                f"关于 '{query}' 的信息点 3"
            ]
        
        return results
    
    def _analyze_information(self, subtask: str, info: List[str]) -> str:
        """使用 QwenLLM 分析信息"""
        if self.llm is not None:
            system_prompt = (
                "你是一个研究分析师。请根据提供的参考信息，对研究子任务进行简洁的分析总结。"
                "回答控制在 150 字以内。"
            )
            context = "\n".join([f"- {item}" for item in info])
            user_msg = f"研究子任务: {subtask}\n\n参考信息:\n{context}"
            analysis = self.llm.chat(user_msg, system_prompt=system_prompt, max_new_tokens=300, temperature=0.7)
            return analysis.strip()
        else:
            # ---- 无模型时的备选方案 ----
            key_points = random.sample(info, min(2, len(info)))
            return "; ".join(key_points)
    
    def generate_report(self, task: ResearchTask) -> str:
        """使用 QwenLLM 生成研究报告"""
        print(f"\n生成研究报告: {task.topic}")
        print("="*60)
        
        if self.llm is not None:
            findings = "\n".join([
                f"### {subtask}\n{result}"
                for subtask, result in task.results.items()
            ])
            system_prompt = (
                "你是一个专业的研究报告撰写专家。请根据研究子任务的发现，"
                "生成一份结构清晰、内容完整的研究报告。"
                "报告应包含：研究概述、研究发现（分章节）、总结与展望。"
            )
            user_msg = f"研究主题: {task.topic}\n\n研究发现:\n{findings}"
            report = self.llm.chat(user_msg, system_prompt=system_prompt, max_new_tokens=1024, temperature=0.7)
            return report.strip()
        else:
            # ---- 无模型时的备选方案 ----
            report = f"""# {task.topic} 研究报告

## 研究概述
本报告对 {task.topic} 进行了系统性研究，涵盖基本概念、核心技术、应用场景和发展趋势。

## 研究发现
"""
            
            for i, (subtask, result) in enumerate(task.results.items(), 1):
                report += f"\n### {i}. {subtask}\n\n{result}\n"
            
            report += f"""

## 总结
{task.topic} 是一个重要的研究领域，具有广泛的应用前景。
建议持续关注该领域的最新进展。
"""
            return report

# 创建研究助手（传入 QwenLLM 实例）
assistant = ResearchAssistant(llm_model=llm)

# 执行研究
topic = "AI Agent 技术"
task = assistant.plan_research(topic)
results = assistant.research(task)
report = assistant.generate_report(task)

print("\n" + "="*60)
print(report)

---

## 3. 项目总结

### 实现的功能

1. ✅ 任务分解和规划
2. ✅ 信息收集和检索
3. ✅ 内容分析和总结
4. ✅ 结构化报告生成

### 可优化方向

- 集成真实搜索引擎 API
- 添加引用和来源验证
- 支持多模态内容分析
- 添加可视化图表生成

---

## 参考

- [AutoGen Research Agent](https://microsoft.github.io/autogen/)
- [CrewAI Research Crew](https://docs.crewai.com/)